
> This notebook contains the functional and maintainability tests that were run, using the testing pipeline we designed. It assess compilation, runtime and test mismatch errors. It also uses Lizard for static code analysis.




# Functional Testing



> Functional testing on XMainframe - compilation, runtime, test mismatch errors


In [ ]:
import os
import json
import subprocess
import shutil
from pathlib import Path
from tqdm import tqdm


# ===== CONFIGURATION =====
TRANSLATED_JAVA_DIR = 'Dissertation/Functional/XMainframe_results'
CODENET_DIR = 'CodeNet_Dataset'
RESULTS_DIR = 'OUTPUT DIR'
RESULTS_FILE = os.path.join(RESULTS_DIR, 'xmainframe_functional_test_result.json')
TEMP_COMPILE_DIR = 'temp_compile'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# ===== TESTING FUNCTIONS =====

def extract_main_class_name(java_file_path):
    """
    Extract the main class name from Java source code

    Returns:
        class_name: str or None
    """
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # Look for: public class ClassName
        import re
        match = re.search(r'public\s+class\s+(\w+)', content)
        if match:
            return match.group(1)

        # Fallback: look for any class declaration
        match = re.search(r'class\s+(\w+)', content)
        if match:
            return match.group(1)

        return None
    except Exception as e:
        return None


def compile_java(java_file_path, problem_id, java_filename):
    """
    Compile a Java file

    Returns:
        (success: bool, error_msg: str or None, class_name: str or None)
    """
    try:
        # Extract the actual class name from the Java file
        class_name = extract_main_class_name(java_file_path)

        if not class_name:
            return False, "Could not extract class name from Java file", None

        # Copy Java file to temp directory with the correct class name
        correct_filename = f"{class_name}.java"
        temp_java_path = os.path.join(TEMP_COMPILE_DIR, correct_filename)
        shutil.copy(java_file_path, temp_java_path)

        # Compile
        result = subprocess.run(
            ['javac', temp_java_path],
            capture_output=True,
            text=True,
            timeout=10,
            cwd=TEMP_COMPILE_DIR
        )

        if result.returncode == 0:
            # Check if .class file was generated
            class_file = os.path.join(TEMP_COMPILE_DIR, f'{class_name}.class')

            if os.path.exists(class_file):
                return True, None, class_name
            else:
                return False, "Compilation succeeded but no .class file found", None
        else:
            return False, result.stderr.strip(), None

    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None


def run_java_with_input(class_name, input_data):
    """
    Run compiled Java class with input

    Returns:
        (success: bool, output: str or None, error_msg: str or None, execution_time: float or None)
    """
    try:
        import time
        start_time = time.time()

        result = subprocess.run(
            ['java', '-cp', TEMP_COMPILE_DIR, class_name],  # ← Added -cp flag
            input=input_data,
            capture_output=True,
            text=True,
            timeout=5
            # Removed cwd parameter
        )

        execution_time = time.time() - start_time

        if result.returncode == 0:
            return True, result.stdout.strip(), None, execution_time
        else:
            return False, None, f"Runtime error: {result.stderr.strip()}", None

    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None



def compare_outputs(actual_output, expected_output):
    """
    Compare actual vs expected output

    Returns:
        (match: bool, details: str)
    """
    # Normalize whitespace
    actual_lines = [line.strip() for line in actual_output.strip().split('\n')]
    expected_lines = [line.strip() for line in expected_output.strip().split('\n')]

    if actual_lines == expected_lines:
        return True, "Output matches expected"
    else:
        details = f"Expected {len(expected_lines)} lines, got {len(actual_lines)} lines.\n"
        details += f"Expected:\n{expected_output[:200]}\n"
        details += f"Actual:\n{actual_output[:200]}"
        return False, details


def clean_temp_dir():
    """Clean up temporary compilation files"""
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
        os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)


def test_single_java_file(problem_id, java_filename, java_file_path):
    """
    Test a single Java file: compile, run, and validate output

    Returns:
        Dictionary with test results
    """
    result = {
        'problem_id': problem_id,
        'java_file': java_filename,
        'class_name': None,
        'compilation': {'success': False, 'error': None},
        'execution': {'success': False, 'error': None, 'output': None, 'time_seconds': None},
        'test_validation': {'success': False, 'error': None},
        'overall_status': 'failed'
    }

    # Step 1: Compile
    compile_success, compile_error, class_name = compile_java(java_file_path, problem_id, java_filename)
    result['compilation']['success'] = compile_success
    result['compilation']['error'] = compile_error
    result['class_name'] = class_name

    if not compile_success:
        return result

    # Step 2: Get test input/output
    problem_dir = os.path.join(CODENET_DIR, problem_id)
    input_file = os.path.join(problem_dir, 'input.txt')
    output_file = os.path.join(problem_dir, 'output.txt')

    if not os.path.exists(input_file) or not os.path.exists(output_file):
        result['execution']['error'] = "Test files (input.txt/output.txt) not found"
        return result

    # Read test data
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            test_input = f.read()
        with open(output_file, 'r', encoding='utf-8') as f:
            expected_output = f.read()
    except Exception as e:
        result['execution']['error'] = f"Error reading test files: {str(e)}"
        return result

    # Step 3: Run with test input
    run_success, actual_output, run_error, exec_time = run_java_with_input(class_name, test_input)
    result['execution']['success'] = run_success
    result['execution']['error'] = run_error
    result['execution']['output'] = actual_output[:500] if actual_output else None  # Limit output length
    result['execution']['time_seconds'] = round(exec_time, 4) if exec_time else None

    if not run_success:
        return result

    # Step 4: Compare outputs
    match, details = compare_outputs(actual_output, expected_output)
    result['test_validation']['success'] = match
    result['test_validation']['error'] = None if match else details

    # Overall status
    if match:
        result['overall_status'] = 'passed'

    return result


def test_all_problems():
    """
    Test all Java files in the translated directory
    """
    print("="*80)
    print("JAVA TRANSLATION TESTING PIPELINE")
    print("="*80)
    print(f"\nTranslated Java: {TRANSLATED_JAVA_DIR}")
    print(f"Test data: {CODENET_DIR}")
    print(f"Results will be saved to: {RESULTS_FILE}\n")

    # Get all problem directories
    problem_dirs = sorted([
        d for d in os.listdir(TRANSLATED_JAVA_DIR)
        if os.path.isdir(os.path.join(TRANSLATED_JAVA_DIR, d))
    ])

    print(f" Found {len(problem_dirs)} problems to test\n")

    all_results = []

    # Statistics
    stats = {
        'total': 0,
        'compiled': 0,
        'executed': 0,
        'passed': 0,
        'compilation_failed': 0,
        'execution_failed': 0,
        'output_mismatch': 0
    }

    for problem_id in tqdm(problem_dirs, desc="Testing"):
        problem_java_dir = os.path.join(TRANSLATED_JAVA_DIR, problem_id)

        # Get all Java files for this problem
        java_files = [f for f in os.listdir(problem_java_dir) if f.endswith('.java')]

        for java_file in java_files:
            stats['total'] += 1
            java_path = os.path.join(problem_java_dir, java_file)

            # Test this Java file
            test_result = test_single_java_file(problem_id, java_file, java_path)
            all_results.append(test_result)

            # Update statistics
            if test_result['compilation']['success']:
                stats['compiled'] += 1

                if test_result['execution']['success']:
                    stats['executed'] += 1

                    if test_result['test_validation']['success']:
                        stats['passed'] += 1
                    else:
                        stats['output_mismatch'] += 1
                else:
                    stats['execution_failed'] += 1
            else:
                stats['compilation_failed'] += 1

            # Clean up after each test
            clean_temp_dir()

    # Save detailed results
    with open(RESULTS_FILE, 'w') as f:
        json.dump({
            'summary': stats,
            'detailed_results': all_results
        }, f, indent=2)

    # Print summary
    print("\n" + "="*80)
    print("TESTING SUMMARY")
    print("="*80)

    total = stats['total']
    print(f"\n Total Java files tested: {total}")
    print(f"\n Compilation success: {stats['compiled']}/{total} ({stats['compiled']/total*100:.1f}%)")
    print(f" Execution success: {stats['executed']}/{total} ({stats['executed']/total*100:.1f}%)")
    print(f" Test passed: {stats['passed']}/{total} ({stats['passed']/total*100:.1f}%)")

    print(f"\n Compilation failed: {stats['compilation_failed']}/{total} ({stats['compilation_failed']/total*100:.1f}%)")
    print(f" Execution failed: {stats['execution_failed']}/{total} ({stats['execution_failed']/total*100:.1f}%)")
    print(f" Output mismatch: {stats['output_mismatch']}/{total} ({stats['output_mismatch']/total*100:.1f}%)")

    # Execution time statistics
    successful_executions = [r for r in all_results if r['execution']['time_seconds'] is not None]
    if successful_executions:
        exec_times = [r['execution']['time_seconds'] for r in successful_executions]
        avg_time = sum(exec_times) / len(exec_times)
        max_time = max(exec_times)
        min_time = min(exec_times)
        print(f"\n  Execution Time Stats:")
        print(f"   Average: {avg_time:.4f}s")
        print(f"   Min: {min_time:.4f}s")
        print(f"   Max: {max_time:.4f}s")

    # Show sample failures
    print("\n" + "="*80)
    print("SAMPLE FAILURES")
    print("="*80)

    # Compilation failures
    comp_failures = [r for r in all_results if not r['compilation']['success']]
    if comp_failures:
        print(f"\n Compilation Failures (showing first 3 of {len(comp_failures)}):")
        for result in comp_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['compilation']['error'][:150]}")

    # Execution failures
    exec_failures = [r for r in all_results if r['compilation']['success'] and not r['execution']['success']]
    if exec_failures:
        print(f"\n Execution Failures (showing first 3 of {len(exec_failures)}):")
        for result in exec_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['execution']['error'][:150]}")

    # Output mismatches
    output_failures = [r for r in all_results if r['execution']['success'] and not r['test_validation']['success']]
    if output_failures:
        print(f"\n Output Mismatches (showing first 3 of {len(output_failures)}):")
        for result in output_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    {result['test_validation']['error'][:150]}")

    print(f"\n Detailed results saved to: {RESULTS_FILE}")
    print("\n Testing complete!")


if __name__ == "__main__":
    test_all_problems()



> Functional testing on Qwen - compilation, runtime, test mismatch errors



In [ ]:
import os
import json
import subprocess
import shutil
from pathlib import Path
from tqdm import tqdm


# ===== CONFIGURATION =====
TRANSLATED_JAVA_DIR = 'Dissertation/Functional/Qwen_results'
CODENET_DIR = 'CodeNet_Dataset'
RESULTS_DIR = 'OUTPUT DIR'
RESULTS_FILE = os.path.join(RESULTS_DIR, 'qwen_functional_test_result.json')
TEMP_COMPILE_DIR = 'temp_compile'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# ===== TESTING FUNCTIONS =====

def extract_main_class_name(java_file_path):
    """
    Extract the main class name from Java source code

    Returns:
        class_name: str or None
    """
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # Look for: public class ClassName
        import re
        match = re.search(r'public\s+class\s+(\w+)', content)
        if match:
            return match.group(1)

        # Fallback: look for any class declaration
        match = re.search(r'class\s+(\w+)', content)
        if match:
            return match.group(1)

        return None
    except Exception as e:
        return None


def compile_java(java_file_path, problem_id, java_filename):
    """
    Compile a Java file

    Returns:
        (success: bool, error_msg: str or None, class_name: str or None)
    """
    try:
        # Extract the actual class name from the Java file
        class_name = extract_main_class_name(java_file_path)

        if not class_name:
            return False, "Could not extract class name from Java file", None

        # Copy Java file to temp directory with the correct class name
        correct_filename = f"{class_name}.java"
        temp_java_path = os.path.join(TEMP_COMPILE_DIR, correct_filename)
        shutil.copy(java_file_path, temp_java_path)

        # Compile
        result = subprocess.run(
            ['javac', temp_java_path],
            capture_output=True,
            text=True,
            timeout=10,
            cwd=TEMP_COMPILE_DIR
        )

        if result.returncode == 0:
            # Check if .class file was generated
            class_file = os.path.join(TEMP_COMPILE_DIR, f'{class_name}.class')

            if os.path.exists(class_file):
                return True, None, class_name
            else:
                return False, "Compilation succeeded but no .class file found", None
        else:
            return False, result.stderr.strip(), None

    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None


def run_java_with_input(class_name, input_data):
    """
    Run compiled Java class with input

    Returns:
        (success: bool, output: str or None, error_msg: str or None, execution_time: float or None)
    """
    try:
        import time
        start_time = time.time()

        result = subprocess.run(
            ['java', '-cp', TEMP_COMPILE_DIR, class_name],  # ← Added -cp flag
            input=input_data,
            capture_output=True,
            text=True,
            timeout=5
            # Removed cwd parameter
        )

        execution_time = time.time() - start_time

        if result.returncode == 0:
            return True, result.stdout.strip(), None, execution_time
        else:
            return False, None, f"Runtime error: {result.stderr.strip()}", None

    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None

def compare_outputs(actual_output, expected_output):
    """
    Compare actual vs expected output

    Returns:
        (match: bool, details: str)
    """
    # Normalize whitespace
    actual_lines = [line.strip() for line in actual_output.strip().split('\n')]
    expected_lines = [line.strip() for line in expected_output.strip().split('\n')]

    if actual_lines == expected_lines:
        return True, "Output matches expected"
    else:
        details = f"Expected {len(expected_lines)} lines, got {len(actual_lines)} lines.\n"
        details += f"Expected:\n{expected_output[:200]}\n"
        details += f"Actual:\n{actual_output[:200]}"
        return False, details


def clean_temp_dir():
    """Clean up temporary compilation files"""
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
        os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)


def test_single_java_file(problem_id, java_filename, java_file_path):
    """
    Test a single Java file: compile, run, and validate output

    Returns:
        Dictionary with test results
    """
    result = {
        'problem_id': problem_id,
        'java_file': java_filename,
        'class_name': None,
        'compilation': {'success': False, 'error': None},
        'execution': {'success': False, 'error': None, 'output': None, 'time_seconds': None},
        'test_validation': {'success': False, 'error': None},
        'overall_status': 'failed'
    }

    # Step 1: Compile
    compile_success, compile_error, class_name = compile_java(java_file_path, problem_id, java_filename)
    result['compilation']['success'] = compile_success
    result['compilation']['error'] = compile_error
    result['class_name'] = class_name

    if not compile_success:
        return result

    # Step 2: Get test input/output
    problem_dir = os.path.join(CODENET_DIR, problem_id)
    input_file = os.path.join(problem_dir, 'input.txt')
    output_file = os.path.join(problem_dir, 'output.txt')

    if not os.path.exists(input_file) or not os.path.exists(output_file):
        result['execution']['error'] = "Test files (input.txt/output.txt) not found"
        return result

    # Read test data
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            test_input = f.read()
        with open(output_file, 'r', encoding='utf-8') as f:
            expected_output = f.read()
    except Exception as e:
        result['execution']['error'] = f"Error reading test files: {str(e)}"
        return result

    # Step 3: Run with test input
    run_success, actual_output, run_error, exec_time = run_java_with_input(class_name, test_input)
    result['execution']['success'] = run_success
    result['execution']['error'] = run_error
    result['execution']['output'] = actual_output[:500] if actual_output else None  # Limit output length
    result['execution']['time_seconds'] = round(exec_time, 4) if exec_time else None

    if not run_success:
        return result

    # Step 4: Compare outputs
    match, details = compare_outputs(actual_output, expected_output)
    result['test_validation']['success'] = match
    result['test_validation']['error'] = None if match else details

    # Overall status
    if match:
        result['overall_status'] = 'passed'

    return result


def test_all_problems():
    """
    Test all Java files in the translated directory
    """
    print("="*80)
    print("JAVA TRANSLATION TESTING PIPELINE")
    print("="*80)
    print(f"\nTranslated Java: {TRANSLATED_JAVA_DIR}")
    print(f"Test data: {CODENET_DIR}")
    print(f"Results will be saved to: {RESULTS_FILE}\n")

    # Get all problem directories
    problem_dirs = sorted([
        d for d in os.listdir(TRANSLATED_JAVA_DIR)
        if os.path.isdir(os.path.join(TRANSLATED_JAVA_DIR, d))
    ])

    print(f" Found {len(problem_dirs)} problems to test\n")

    all_results = []

    # Statistics
    stats = {
        'total': 0,
        'compiled': 0,
        'executed': 0,
        'passed': 0,
        'compilation_failed': 0,
        'execution_failed': 0,
        'output_mismatch': 0
    }

    for problem_id in tqdm(problem_dirs, desc="Testing"):
        problem_java_dir = os.path.join(TRANSLATED_JAVA_DIR, problem_id)

        # Get all Java files for this problem
        java_files = [f for f in os.listdir(problem_java_dir) if f.endswith('.java')]

        for java_file in java_files:
            stats['total'] += 1
            java_path = os.path.join(problem_java_dir, java_file)

            # Test this Java file
            test_result = test_single_java_file(problem_id, java_file, java_path)
            all_results.append(test_result)

            # Update statistics
            if test_result['compilation']['success']:
                stats['compiled'] += 1

                if test_result['execution']['success']:
                    stats['executed'] += 1

                    if test_result['test_validation']['success']:
                        stats['passed'] += 1
                    else:
                        stats['output_mismatch'] += 1
                else:
                    stats['execution_failed'] += 1
            else:
                stats['compilation_failed'] += 1

            # Clean up after each test
            clean_temp_dir()

    # Save detailed results
    with open(RESULTS_FILE, 'w') as f:
        json.dump({
            'summary': stats,
            'detailed_results': all_results
        }, f, indent=2)

    # Print summary
    print("\n" + "="*80)
    print("TESTING SUMMARY")
    print("="*80)

    total = stats['total']
    print(f"\n Total Java files tested: {total}")
    print(f"\n Compilation success: {stats['compiled']}/{total} ({stats['compiled']/total*100:.1f}%)")
    print(f" Execution success: {stats['executed']}/{total} ({stats['executed']/total*100:.1f}%)")
    print(f" Test passed: {stats['passed']}/{total} ({stats['passed']/total*100:.1f}%)")

    print(f"\n Compilation failed: {stats['compilation_failed']}/{total} ({stats['compilation_failed']/total*100:.1f}%)")
    print(f" Execution failed: {stats['execution_failed']}/{total} ({stats['execution_failed']/total*100:.1f}%)")
    print(f" Output mismatch: {stats['output_mismatch']}/{total} ({stats['output_mismatch']/total*100:.1f}%)")

    # Execution time statistics
    successful_executions = [r for r in all_results if r['execution']['time_seconds'] is not None]
    if successful_executions:
        exec_times = [r['execution']['time_seconds'] for r in successful_executions]
        avg_time = sum(exec_times) / len(exec_times)
        max_time = max(exec_times)
        min_time = min(exec_times)
        print(f"\n Execution Time Stats:")
        print(f"   Average: {avg_time:.4f}s")
        print(f"   Min: {min_time:.4f}s")
        print(f"   Max: {max_time:.4f}s")

    # Show sample failures
    print("\n" + "="*80)
    print("SAMPLE FAILURES")
    print("="*80)

    # Compilation failures
    comp_failures = [r for r in all_results if not r['compilation']['success']]
    if comp_failures:
        print(f"\n Compilation Failures (showing first 3 of {len(comp_failures)}):")
        for result in comp_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['compilation']['error'][:150]}")

    # Execution failures
    exec_failures = [r for r in all_results if r['compilation']['success'] and not r['execution']['success']]
    if exec_failures:
        print(f"\n Execution Failures (showing first 3 of {len(exec_failures)}):")
        for result in exec_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['execution']['error'][:150]}")

    # Output mismatches
    output_failures = [r for r in all_results if r['execution']['success'] and not r['test_validation']['success']]
    if output_failures:
        print(f"\n Output Mismatches (showing first 3 of {len(output_failures)}):")
        for result in output_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    {result['test_validation']['error'][:150]}")

    print(f"\n Detailed results saved to: {RESULTS_FILE}")
    print("\n Testing complete!")


if __name__ == "__main__":
    test_all_problems()



> Functional testing on COBOL4J - compilation, runtime, test mismatch errors



In [ ]:
import os
import subprocess

# ===== INSTALL OPENSOURCECOBOL4J =====

print("="*80)
print("INSTALLING OPENSOURCECOBOL4J")
print("="*80)

# Install dependencies
print("\n Installing dependencies...")
os.system("apt-get update -qq")
os.system("apt-get install -y -qq default-jdk build-essential bison flex gettext texinfo libgmp-dev autoconf")

# Download and install opensourcecobol4j
print("\n Downloading opensourcecobol4j v1.1.16...")
os.system("curl -L -o opensourcecobol4j-v1.1.16.tar.gz https://github.com/opensourcecobol/opensourcecobol4j/archive/refs/tags/v1.1.16.tar.gz")

print("\n Extracting...")
os.system("tar zxf opensourcecobol4j-v1.1.16.tar.gz")

print("\n Configuring and building...")
os.chdir("opensourcecobol4j-1.1.16")
os.system("./configure --prefix=/usr/")
os.system("make")
os.system("make install")

# Set CLASSPATH
print("\nSetting CLASSPATH...")
os.environ['CLASSPATH'] = '/usr/lib/opensourcecobol4j/libcobj.jar'

# Return to original directory
os.chdir("/content")

print("\n OpenSourceCOBOL4J installed successfully!")

# Verify installation
result = subprocess.run(['cobj', '--version'], capture_output=True, text=True)
print(f"\n{result.stdout}")



In [ ]:
import os
import json
import subprocess
import shutil
from pathlib import Path
from tqdm import tqdm



# ===== CONFIGURATION =====
TRANSLATED_JAVA_DIR = 'Dissertation/Functional/COBOL4J_results'
CODENET_DIR = 'CodeNet_Dataset'
RESULTS_DIR = 'OUTPUT_DIR'
RESULTS_FILE = os.path.join(RESULTS_DIR, 'COBOL4J_functional_test_result.json')
TEMP_COMPILE_DIR = 'temp_compile'


# OpenSourceCOBOL4J specific
LIBCOBJ_JAR = '/usr/lib/opensourcecobol4j/libcobj.jar'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)

# Verify libcobj.jar exists
if not os.path.exists(LIBCOBJ_JAR):
    print(f" ERROR: {LIBCOBJ_JAR} not found!")
    print("Please install OpenSourceCOBOL4J first")
    exit(1)

print(f" Found libcobj.jar: {LIBCOBJ_JAR}\n")

# ===== TESTING FUNCTIONS =====

def extract_main_class_name(java_file_path):
    """
    Extract the main class name from Java source code

    Returns:
        class_name: str or None
    """
    try:
        with open(java_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # Look for: public class ClassName
        import re
        match = re.search(r'public\s+class\s+(\w+)', content)
        if match:
            return match.group(1)

        # Fallback: look for any class declaration
        match = re.search(r'class\s+(\w+)', content)
        if match:
            return match.group(1)

        return None
    except Exception as e:
        return None


def compile_java(java_file_path, problem_id, java_filename):
    """
    Compile a Java file with OpenSourceCOBOL4J runtime

    Returns:
        (success: bool, error_msg: str or None, class_name: str or None)
    """
    try:
        # Extract the actual class name from the Java file
        class_name = extract_main_class_name(java_file_path)

        if not class_name:
            return False, "Could not extract class name from Java file", None

        # Copy Java file to temp directory with the correct class name
        correct_filename = f"{class_name}.java"
        temp_java_path = os.path.join(TEMP_COMPILE_DIR, correct_filename)
        shutil.copy(java_file_path, temp_java_path)

        # Compile with libcobj.jar in classpath
        result = subprocess.run(
            ['javac', '-cp', LIBCOBJ_JAR, temp_java_path],
            capture_output=True,
            text=True,
            timeout=10,
            cwd=TEMP_COMPILE_DIR
        )

        if result.returncode == 0:
            # Check if .class file was generated
            class_file = os.path.join(TEMP_COMPILE_DIR, f'{class_name}.class')

            if os.path.exists(class_file):
                return True, None, class_name
            else:
                return False, "Compilation succeeded but no .class file found", None
        else:
            return False, result.stderr.strip(), None

    except subprocess.TimeoutExpired:
        return False, "Compilation timeout (>10s)", None
    except Exception as e:
        return False, str(e), None


def run_java_with_input(class_name, input_data):
    """
    Run compiled Java class with input (with libcobj.jar in classpath)

    Returns:
        (success: bool, output: str or None, error_msg: str or None, execution_time: float or None)
    """
    try:
        import time
        start_time = time.time()

        # Run with libcobj.jar in classpath
        result = subprocess.run(
            ['java', '-cp', f'.:{LIBCOBJ_JAR}', class_name],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=5,
            cwd=TEMP_COMPILE_DIR
        )

        execution_time = time.time() - start_time

        if result.returncode == 0:
            return True, result.stdout.strip(), None, execution_time
        else:
            return False, None, f"Runtime error: {result.stderr.strip()}", None

    except subprocess.TimeoutExpired:
        return False, None, "Execution timeout (>5s)", None
    except Exception as e:
        return False, None, str(e), None


def compare_outputs(actual_output, expected_output):
    """
    Compare actual vs expected output

    Returns:
        (match: bool, details: str)
    """
    # Normalize whitespace
    actual_lines = [line.strip() for line in actual_output.strip().split('\n')]
    expected_lines = [line.strip() for line in expected_output.strip().split('\n')]

    if actual_lines == expected_lines:
        return True, "Output matches expected"
    else:
        details = f"Expected {len(expected_lines)} lines, got {len(actual_lines)} lines.\n"
        details += f"Expected:\n{expected_output[:200]}\n"
        details += f"Actual:\n{actual_output[:200]}"
        return False, details


def clean_temp_dir():
    """Clean up temporary compilation files"""
    if os.path.exists(TEMP_COMPILE_DIR):
        shutil.rmtree(TEMP_COMPILE_DIR)
        os.makedirs(TEMP_COMPILE_DIR, exist_ok=True)


def test_single_java_file(problem_id, java_filename, java_file_path):
    """
    Test a single Java file: compile, run, and validate output

    Returns:
        Dictionary with test results
    """
    result = {
        'problem_id': problem_id,
        'java_file': java_filename,
        'class_name': None,
        'compilation': {'success': False, 'error': None},
        'execution': {'success': False, 'error': None, 'output': None, 'time_seconds': None},
        'test_validation': {'success': False, 'error': None},
        'overall_status': 'failed'
    }

    # Step 1: Compile
    compile_success, compile_error, class_name = compile_java(java_file_path, problem_id, java_filename)
    result['compilation']['success'] = compile_success
    result['compilation']['error'] = compile_error
    result['class_name'] = class_name

    if not compile_success:
        return result

    # Step 2: Get test input/output
    problem_dir = os.path.join(CODENET_DIR, problem_id)
    input_file = os.path.join(problem_dir, 'input.txt')
    output_file = os.path.join(problem_dir, 'output.txt')

    if not os.path.exists(input_file) or not os.path.exists(output_file):
        result['execution']['error'] = "Test files (input.txt/output.txt) not found"
        return result

    # Read test data
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            test_input = f.read()
        with open(output_file, 'r', encoding='utf-8') as f:
            expected_output = f.read()
    except Exception as e:
        result['execution']['error'] = f"Error reading test files: {str(e)}"
        return result

    # Step 3: Run with test input
    run_success, actual_output, run_error, exec_time = run_java_with_input(class_name, test_input)
    result['execution']['success'] = run_success
    result['execution']['error'] = run_error
    result['execution']['output'] = actual_output[:500] if actual_output else None
    result['execution']['time_seconds'] = round(exec_time, 4) if exec_time else None

    if not run_success:
        return result

    # Step 4: Compare outputs
    match, details = compare_outputs(actual_output, expected_output)
    result['test_validation']['success'] = match
    result['test_validation']['error'] = None if match else details

    # Overall status
    if match:
        result['overall_status'] = 'passed'

    return result


def test_all_problems():
    """
    Test all Java files in the translated directory
    """
    print("="*80)
    print("NON-LLM (OpenSourceCOBOL4J) TESTING PIPELINE")
    print("="*80)
    print(f"\nTranslated Java: {TRANSLATED_JAVA_DIR}")
    print(f"Test data: {CODENET_DIR}")
    print(f"Results will be saved to: {RESULTS_FILE}\n")

    # Get all problem directories
    problem_dirs = sorted([
        d for d in os.listdir(TRANSLATED_JAVA_DIR)
        if os.path.isdir(os.path.join(TRANSLATED_JAVA_DIR, d))
    ])

    print(f" Found {len(problem_dirs)} problems to test\n")

    all_results = []

    # Statistics
    stats = {
        'total': 0,
        'compiled': 0,
        'executed': 0,
        'passed': 0,
        'compilation_failed': 0,
        'execution_failed': 0,
        'output_mismatch': 0
    }

    for problem_id in tqdm(problem_dirs, desc="Testing"):
        problem_java_dir = os.path.join(TRANSLATED_JAVA_DIR, problem_id)

        # Get all Java files for this problem
        java_files = [f for f in os.listdir(problem_java_dir) if f.endswith('.java')]

        for java_file in java_files:
            stats['total'] += 1
            java_path = os.path.join(problem_java_dir, java_file)

            # Test this Java file
            test_result = test_single_java_file(problem_id, java_file, java_path)
            all_results.append(test_result)

            # Update statistics
            if test_result['compilation']['success']:
                stats['compiled'] += 1

                if test_result['execution']['success']:
                    stats['executed'] += 1

                    if test_result['test_validation']['success']:
                        stats['passed'] += 1
                    else:
                        stats['output_mismatch'] += 1
                else:
                    stats['execution_failed'] += 1
            else:
                stats['compilation_failed'] += 1

            # Clean up after each test
            clean_temp_dir()

    # Save detailed results
    with open(RESULTS_FILE, 'w') as f:
        json.dump({
            'summary': stats,
            'detailed_results': all_results
        }, f, indent=2)

    # Print summary
    print("\n" + "="*80)
    print("TESTING SUMMARY")
    print("="*80)

    total = stats['total']
    print(f"\n Total Java files tested: {total}")
    print(f"\n Compilation success: {stats['compiled']}/{total} ({stats['compiled']/total*100:.1f}%)")
    print(f" Execution success: {stats['executed']}/{total} ({stats['executed']/total*100:.1f}%)")
    print(f" Test passed: {stats['passed']}/{total} ({stats['passed']/total*100:.1f}%)")

    print(f"\n Compilation failed: {stats['compilation_failed']}/{total} ({stats['compilation_failed']/total*100:.1f}%)")
    print(f" Execution failed: {stats['execution_failed']}/{total} ({stats['execution_failed']/total*100:.1f}%)")
    print(f" Output mismatch: {stats['output_mismatch']}/{total} ({stats['output_mismatch']/total*100:.1f}%)")

    # Execution time statistics
    successful_executions = [r for r in all_results if r['execution']['time_seconds'] is not None]
    if successful_executions:
        exec_times = [r['execution']['time_seconds'] for r in successful_executions]
        avg_time = sum(exec_times) / len(exec_times)
        max_time = max(exec_times)
        min_time = min(exec_times)
        print(f"\n  Execution Time Stats:")
        print(f"   Average: {avg_time:.4f}s")
        print(f"   Min: {min_time:.4f}s")
        print(f"   Max: {max_time:.4f}s")

    # Show sample failures
    print("\n" + "="*80)
    print("SAMPLE FAILURES")
    print("="*80)

    # Compilation failures
    comp_failures = [r for r in all_results if not r['compilation']['success']]
    if comp_failures:
        print(f"\n Compilation Failures (showing first 3 of {len(comp_failures)}):")
        for result in comp_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['compilation']['error'][:150]}")

    # Execution failures
    exec_failures = [r for r in all_results if r['compilation']['success'] and not r['execution']['success']]
    if exec_failures:
        print(f"\n Execution Failures (showing first 3 of {len(exec_failures)}):")
        for result in exec_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    Error: {result['execution']['error'][:150]}")

    # Output mismatches
    output_failures = [r for r in all_results if r['execution']['success'] and not r['test_validation']['success']]
    if output_failures:
        print(f"\n Output Mismatches (showing first 3 of {len(output_failures)}):")
        for result in output_failures[:3]:
            print(f"\n  {result['problem_id']}/{result['java_file']}:")
            print(f"    {result['test_validation']['error'][:150]}")

    print(f"\n Detailed results saved to: {RESULTS_FILE}")
    print("\n Testing complete!")


if __name__ == "__main__":
    test_all_problems()

# Lizard Static Code Analysis

We are only considering translations that are at least compiling (ignoring compilation failures) and additionally documenting the files that Lizard rejected.



> Static Analysis on Xmainframe translations




In [ ]:

import os
import json
import subprocess
from tqdm import tqdm

print("Installing Lizard...")
os.system("pip install -q lizard")

SOURCE_DIR = 'Dissertation/Functional/XMainframe_results' # where the translated Java files are located
OUTPUT_DIR = 'OUTPUT_DIR'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'static_analysis_xmainframe.json')
COMPILATION_RESULTS_FILE = 'Dissertation/Functional/jsons_functional_testing/xmainframe_after_clean_result.json' # which has the data on files that failed compilation

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80) 
print("LIZARD STATIC ANALYSIS - XMAINFRAME")
print("="*80)

# Load compilation results
with open(COMPILATION_RESULTS_FILE, 'r') as f:
    compilation_data = json.load(f)

failed_files = set()
for entry in compilation_data.get('detailed_results', []):
    if not entry.get('compilation', {}).get('success', True):
        failed_files.add((entry['problem_id'], entry['java_file']))

problem_dirs = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

# Pre-scan counts
total_java_files = 0
for pid in problem_dirs:
    problem_dir = os.path.join(SOURCE_DIR, pid)
    total_java_files += len([f for f in os.listdir(problem_dir) if f.endswith('.java')])

all_metrics = []
overall_stats = {
    'total_files': 0,
    'total_nloc': 0,
    'total_ccn': 0,
    'total_token': 0,
    'total_functions': 0,
    'avg_nloc_per_function': 0,
    'avg_ccn_per_function': 0,
    'avg_token_per_function': 0,
    'max_ccn': 0,
    'max_nloc': 0,
}

count_compiled_failed = 0
count_lizard_failed = 0
count_analyzed = 0

for pid in tqdm(problem_dirs, desc="Analyzing"):
    problem_dir = os.path.join(SOURCE_DIR, pid)
    all_java = [f for f in os.listdir(problem_dir) if f.endswith('.java')]

    for java_file in all_java:
        # Skip compilation failures
        if (pid, java_file) in failed_files:
            count_compiled_failed += 1
            continue

        java_path = os.path.join(problem_dir, java_file)

        try:
            result = subprocess.run(
                ['lizard', java_path],
                capture_output=True,
                text=True,
                timeout=30
            )

            if result.returncode != 0 or not result.stdout.strip():
              count_lizard_failed += 1
              continue

            lines = result.stdout.strip().split('\n')
            file_metrics = {
                'problem_id': pid,
                'java_file': java_file,
                'functions': []
            }

            for line in lines:
                if '@' in line and not line.startswith('=') and not line.startswith('-') and 'NLOC' not in line:
                    parts = line.split()
                    if len(parts) >= 6:
                        try:
                            nloc    = int(parts[0])
                            ccn     = int(parts[1])
                            token   = int(parts[2])
                            param   = int(parts[3])
                            length  = int(parts[4])
                            location = ' '.join(parts[5:])

                            function_metric = {
                                'nloc': nloc, 'ccn': ccn, 'token': token,
                                'param': param, 'length': length, 'location': location
                            }
                            file_metrics['functions'].append(function_metric)

                            overall_stats['total_nloc']  += nloc
                            overall_stats['total_ccn']   += ccn
                            overall_stats['total_token'] += token
                            overall_stats['total_functions'] += 1

                            if ccn  > overall_stats['max_ccn']:  overall_stats['max_ccn']  = ccn
                            if nloc > overall_stats['max_nloc']: overall_stats['max_nloc'] = nloc

                        except (ValueError, IndexError):
                            continue

            if file_metrics['functions']:
                file_metrics['file_summary'] = {
                    'total_nloc':          sum(f['nloc'] for f in file_metrics['functions']),
                    'total_ccn':           sum(f['ccn']  for f in file_metrics['functions']),
                    'avg_ccn':             sum(f['ccn']  for f in file_metrics['functions']) / len(file_metrics['functions']),
                    'max_ccn':             max(f['ccn']  for f in file_metrics['functions']),
                    'function_count':      len(file_metrics['functions']),
                    'avg_function_length': sum(f['nloc'] for f in file_metrics['functions']) / len(file_metrics['functions'])
                }
                all_metrics.append(file_metrics)
                overall_stats['total_files'] += 1
                count_analyzed += 1
            else:
                count_lizard_failed += 1

        except subprocess.TimeoutExpired:
            count_lizard_failed += 1
        except Exception:
            count_lizard_failed += 1

# Calculate averages
if overall_stats['total_functions'] > 0:
    overall_stats['avg_nloc_per_function']   = overall_stats['total_nloc']  / overall_stats['total_functions']
    overall_stats['avg_ccn_per_function']    = overall_stats['total_ccn']   / overall_stats['total_functions']
    overall_stats['avg_token_per_function']  = overall_stats['total_token'] / overall_stats['total_functions']

results = {
    'translator': 'XMainframe',
    'overall_statistics': overall_stats,
    'per_file_metrics': all_metrics
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

# ── Final summary ────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("OVERALL STATISTICS ")
print("="*80)

print(f"\n File Accounting (should sum to {total_java_files}):")
print(f"  Successfully analyzed:       {count_analyzed}")
print(f"   Failed compilation:          {count_compiled_failed}")
print(f"   Lizard rejected: {count_lizard_failed}")
print(f"    Total:                       {count_analyzed + count_compiled_failed + count_lizard_failed}")

print(f"\n Functions found: {overall_stats['total_functions']}")


print(f"\n Code Size:")
print(f"   Total NLOC:            {overall_stats['total_nloc']:,}")
print(f"   Avg NLOC per function: {overall_stats['avg_nloc_per_function']:.1f}")
print(f"   Max NLOC:              {overall_stats['max_nloc']}")

print(f"\n Complexity:")
print(f"   Avg CCN per function:  {overall_stats['avg_ccn_per_function']:.2f}")
print(f"   Max CCN:               {overall_stats['max_ccn']}")

print(f"\n Tokens:")
print(f"   Total tokens:          {overall_stats['total_token']:,}")
print(f"   Avg tokens per func:   {overall_stats['avg_token_per_function']:.1f}")

if all_metrics:
    all_ccns = [f['ccn'] for fm in all_metrics for f in fm['functions']]
    if all_ccns:
        low    = sum(1 for c in all_ccns if c <= 10)
        medium = sum(1 for c in all_ccns if 10 < c <= 20)
        high   = sum(1 for c in all_ccns if c > 20)
        print(f"\n Complexity Distribution:")
        print(f"   Low    (CCN ≤ 10):      {low}  ({low/len(all_ccns)*100:.1f}%)")
        print(f"   Medium (10 < CCN ≤ 20): {medium}  ({medium/len(all_ccns)*100:.1f}%)")
        print(f"   High   (CCN > 20):      {high}  ({high/len(all_ccns)*100:.1f}%)")

print(f"\n Results saved to: {OUTPUT_FILE}")
print(" Analysis complete!")



> Static Analysis on Qwen translations




In [ ]:

import os
import json
import subprocess
from tqdm import tqdm

print("Installing Lizard...")
os.system("pip install -q lizard")


SOURCE_DIR = 'Dissertation/Functional/Qwen_after_clean_results'
OUTPUT_DIR = 'OUTPUT DIR'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'static_analysis_qwen.json')
COMPILATION_RESULTS_FILE = 'Dissertation/Functional/jsons_functional_testing/qwen_after_clean_result.json' # which has the data on files that failed compilation

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("LIZARD STATIC ANALYSIS - QWEN")
print("="*80)

# Load compilation results
with open(COMPILATION_RESULTS_FILE, 'r') as f:
    compilation_data = json.load(f)

failed_files = set()
for entry in compilation_data.get('detailed_results', []):
    if not entry.get('compilation', {}).get('success', True):
        failed_files.add((entry['problem_id'], entry['java_file']))

problem_dirs = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

# Pre-scan counts
total_java_files = 0
for pid in problem_dirs:
    problem_dir = os.path.join(SOURCE_DIR, pid)
    total_java_files += len([f for f in os.listdir(problem_dir) if f.endswith('.java')])

all_metrics = []
overall_stats = {
    'total_files': 0,
    'total_nloc': 0,
    'total_ccn': 0,
    'total_token': 0,
    'total_functions': 0,
    'avg_nloc_per_function': 0,
    'avg_ccn_per_function': 0,
    'avg_token_per_function': 0,
    'max_ccn': 0,
    'max_nloc': 0,
}

count_compiled_failed = 0
count_lizard_failed = 0
count_analyzed = 0

for pid in tqdm(problem_dirs, desc="Analyzing"):
    problem_dir = os.path.join(SOURCE_DIR, pid)
    all_java = [f for f in os.listdir(problem_dir) if f.endswith('.java')]

    for java_file in all_java:
        # Skip compilation failures
        if (pid, java_file) in failed_files:
            count_compiled_failed += 1
            continue

        java_path = os.path.join(problem_dir, java_file)

        try:
            result = subprocess.run(
                ['lizard', java_path],
                capture_output=True,
                text=True,
                timeout=30
            )

            if result.returncode != 0 or not result.stdout.strip():
              count_lizard_failed += 1
              continue

            lines = result.stdout.strip().split('\n')
            file_metrics = {
                'problem_id': pid,
                'java_file': java_file,
                'functions': []
            }

            for line in lines:
                if '@' in line and not line.startswith('=') and not line.startswith('-') and 'NLOC' not in line:
                    parts = line.split()
                    if len(parts) >= 6:
                        try:
                            nloc    = int(parts[0])
                            ccn     = int(parts[1])
                            token   = int(parts[2])
                            param   = int(parts[3])
                            length  = int(parts[4])
                            location = ' '.join(parts[5:])

                            function_metric = {
                                'nloc': nloc, 'ccn': ccn, 'token': token,
                                'param': param, 'length': length, 'location': location
                            }
                            file_metrics['functions'].append(function_metric)

                            overall_stats['total_nloc']  += nloc
                            overall_stats['total_ccn']   += ccn
                            overall_stats['total_token'] += token
                            overall_stats['total_functions'] += 1

                            if ccn  > overall_stats['max_ccn']:  overall_stats['max_ccn']  = ccn
                            if nloc > overall_stats['max_nloc']: overall_stats['max_nloc'] = nloc

                        except (ValueError, IndexError):
                            continue

            if file_metrics['functions']:
                file_metrics['file_summary'] = {
                    'total_nloc':          sum(f['nloc'] for f in file_metrics['functions']),
                    'total_ccn':           sum(f['ccn']  for f in file_metrics['functions']),
                    'avg_ccn':             sum(f['ccn']  for f in file_metrics['functions']) / len(file_metrics['functions']),
                    'max_ccn':             max(f['ccn']  for f in file_metrics['functions']),
                    'function_count':      len(file_metrics['functions']),
                    'avg_function_length': sum(f['nloc'] for f in file_metrics['functions']) / len(file_metrics['functions'])
                }
                all_metrics.append(file_metrics)
                overall_stats['total_files'] += 1
                count_analyzed += 1
            else:
                count_lizard_failed += 1

        except subprocess.TimeoutExpired:
            count_lizard_failed += 1
        except Exception:
            count_lizard_failed += 1

# Calculate averages
if overall_stats['total_functions'] > 0:
    overall_stats['avg_nloc_per_function']   = overall_stats['total_nloc']  / overall_stats['total_functions']
    overall_stats['avg_ccn_per_function']    = overall_stats['total_ccn']   / overall_stats['total_functions']
    overall_stats['avg_token_per_function']  = overall_stats['total_token'] / overall_stats['total_functions']

results = {
    'translator': 'qwen',
    'overall_statistics': overall_stats,
    'per_file_metrics': all_metrics
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

# ── Final summary ────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("OVERALL STATISTICS ")
print("="*80)

print(f"\n File Accounting (should sum to {total_java_files}):")
print(f"  Successfully analyzed:       {count_analyzed}")
print(f"   Failed compilation:          {count_compiled_failed}")
print(f"   Lizard rejected: {count_lizard_failed}")
print(f"   Total:                       {count_analyzed + count_compiled_failed + count_lizard_failed}")

print(f"\n Functions found: {overall_stats['total_functions']}")


print(f"\n Code Size:")
print(f"   Total NLOC:            {overall_stats['total_nloc']:,}")
print(f"   Avg NLOC per function: {overall_stats['avg_nloc_per_function']:.1f}")
print(f"   Max NLOC:              {overall_stats['max_nloc']}")

print(f"\n Complexity:")
print(f"   Avg CCN per function:  {overall_stats['avg_ccn_per_function']:.2f}")
print(f"   Max CCN:               {overall_stats['max_ccn']}")

print(f"\n Tokens:")
print(f"   Total tokens:          {overall_stats['total_token']:,}")
print(f"   Avg tokens per func:   {overall_stats['avg_token_per_function']:.1f}")

if all_metrics:
    all_ccns = [f['ccn'] for fm in all_metrics for f in fm['functions']]
    if all_ccns:
        low    = sum(1 for c in all_ccns if c <= 10)
        medium = sum(1 for c in all_ccns if 10 < c <= 20)
        high   = sum(1 for c in all_ccns if c > 20)
        print(f"\n Complexity Distribution:")
        print(f"   Low    (CCN ≤ 10):      {low}  ({low/len(all_ccns)*100:.1f}%)")
        print(f"   Medium (10 < CCN ≤ 20): {medium}  ({medium/len(all_ccns)*100:.1f}%)")
        print(f"   High   (CCN > 20):      {high}  ({high/len(all_ccns)*100:.1f}%)")

print(f"\n Results saved to: {OUTPUT_FILE}")
print(" Analysis complete!")



> Static Analysis on COBOL4J translations


In [ ]:

import os
import json
import subprocess
from tqdm import tqdm

# Install Lizard
print("Installing Lizard...")
os.system("pip install -q lizard")

# Configuration
SOURCE_DIR = 'Dissertation/Functional/COBOL4J_results'
OUTPUT_DIR = 'OUTPUT DIR'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'static_analysis_cobol4j.json')
COMPILATION_RESULTS_FILE = 'Dissertation/Functional/jsons_functional_testing/cobol4j_functional_test_result.json' # which has the data on files that failed compilation

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("LIZARD STATIC ANALYSIS - COBOL4J")
print("="*80)
print(f"\nSource: {SOURCE_DIR}")
print(f"Output: {OUTPUT_FILE}\n")

# ── Load compilation results and build a set of files to SKIP ──────────
print(" Loading compilation results...")
with open(COMPILATION_RESULTS_FILE, 'r') as f:
    compilation_data = json.load(f)

# Build set of (problem_id, java_file) pairs that failed compilation
failed_files = set()
for entry in compilation_data.get('detailed_results', []):
    if not entry.get('compilation', {}).get('success', True):
        failed_files.add((entry['problem_id'], entry['java_file']))

print(f"   Skipping {len(failed_files)} files with compilation errors\n")

# ── Get all problem directories ──────────
problem_dirs = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

print(f" Found {len(problem_dirs)} problems\n")

# ── Prepare metrics containers ──────────
all_metrics = []
overall_stats = {
    'total_files': 0,
    'total_nloc': 0,
    'total_ccn': 0,
    'total_token': 0,
    'total_functions': 0,
    'avg_nloc_per_function': 0,
    'avg_ccn_per_function': 0,
    'avg_token_per_function': 0,
    'max_ccn': 0,
    'max_nloc': 0,
    'skipped_files': len(failed_files)  # track compilation skips
}

lizard_skipped_files = []  # Track files skipped by Lizard (no functions/parse errors)

# ── Main analysis loop ──────────
for pid in tqdm(problem_dirs, desc="Analyzing"):
    problem_dir = os.path.join(SOURCE_DIR, pid)

    # Get Java files, excluding those with compilation errors
    java_files = [
        f for f in os.listdir(problem_dir)
        if f.endswith('.java') and (pid, f) not in failed_files
    ]

    if not java_files:
        continue

    for java_file in java_files:
        java_path = os.path.join(problem_dir, java_file)

        try:
            result = subprocess.run(
                ['lizard', java_path],
                capture_output=True,
                text=True,
                timeout=30
            )

            if result.returncode != 0:
                continue

            lines = result.stdout.strip().split('\n')

            file_metrics = {
                'problem_id': pid,
                'java_file': java_file,
                'functions': []
            }

            for line in lines:
                if '@' in line and not line.startswith('=') and not line.startswith('-') and 'NLOC' not in line:
                    parts = line.split()
                    if len(parts) >= 6:
                        try:
                            nloc = int(parts[0])
                            ccn = int(parts[1])
                            token = int(parts[2])
                            param = int(parts[3])
                            length = int(parts[4])
                            location = ' '.join(parts[5:])

                            function_metric = {
                                'nloc': nloc,
                                'ccn': ccn,
                                'token': token,
                                'param': param,
                                'length': length,
                                'location': location
                            }

                            file_metrics['functions'].append(function_metric)

                            overall_stats['total_nloc'] += nloc
                            overall_stats['total_ccn'] += ccn
                            overall_stats['total_token'] += token
                            overall_stats['total_functions'] += 1

                            if ccn > overall_stats['max_ccn']:
                                overall_stats['max_ccn'] = ccn
                            if nloc > overall_stats['max_nloc']:
                                overall_stats['max_nloc'] = nloc

                        except (ValueError, IndexError):
                            continue

            if file_metrics['functions']:
                file_metrics['file_summary'] = {
                    'total_nloc': sum(f['nloc'] for f in file_metrics['functions']),
                    'total_ccn': sum(f['ccn'] for f in file_metrics['functions']),
                    'avg_ccn': sum(f['ccn'] for f in file_metrics['functions']) / len(file_metrics['functions']),
                    'max_ccn': max(f['ccn'] for f in file_metrics['functions']),
                    'function_count': len(file_metrics['functions']),
                    'avg_function_length': sum(f['nloc'] for f in file_metrics['functions']) / len(file_metrics['functions'])
                }

                all_metrics.append(file_metrics)
                overall_stats['total_files'] += 1
            else:
                # Track files skipped silently by Lizard
                lizard_skipped_files.append((pid, java_file))

        except subprocess.TimeoutExpired:
            print(f" Timeout analyzing {pid}/{java_file}")
        except Exception as e:
            print(f" Error analyzing {pid}/{java_file}: {str(e)[:50]}")

# ── Calculate overall averages ──────────
if overall_stats['total_functions'] > 0:
    overall_stats['avg_nloc_per_function'] = overall_stats['total_nloc'] / overall_stats['total_functions']
    overall_stats['avg_ccn_per_function'] = overall_stats['total_ccn'] / overall_stats['total_functions']
    overall_stats['avg_token_per_function'] = overall_stats['total_token'] / overall_stats['total_functions']

# ── Save results ──────────
results = {
    'translator': 'COBOL4J',
    'overall_statistics': overall_stats,
    'per_file_metrics': all_metrics,
    'lizard_skipped_files': lizard_skipped_files  # New key for skipped files
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

# ── Print summary ──────────
print("\n" + "="*80)
print("OVERALL STATISTICS - COBOL4J")
print("="*80)

print(f"\n Files analyzed:  {overall_stats['total_files']}")
print(f" Files skipped (compilation errors): {overall_stats['skipped_files']}")
print(f"  Files skipped by Lizard (no functions/parse issues): {len(lizard_skipped_files)}")
if lizard_skipped_files:
    print("   Examples:", lizard_skipped_files[:5])

print(f" Total functions: {overall_stats['total_functions']}")

print(f"\n Code Size:")
print(f"   Total NLOC: {overall_stats['total_nloc']:,}")
print(f"   Avg NLOC per function: {overall_stats['avg_nloc_per_function']:.1f}")
print(f"   Max NLOC: {overall_stats['max_nloc']}")

print(f"\n Complexity:")
print(f"   Total CCN: {overall_stats['total_ccn']}")
print(f"   Avg CCN per function: {overall_stats['avg_ccn_per_function']:.2f}")
print(f"   Max CCN: {overall_stats['max_ccn']}")

print(f"\n Tokens:")
print(f"   Total tokens: {overall_stats['total_token']:,}")
print(f"   Avg tokens per function: {overall_stats['avg_token_per_function']:.1f}")

if all_metrics:
    all_ccns = [func['ccn'] for file_metric in all_metrics for func in file_metric['functions']]

    if all_ccns:
        low    = sum(1 for ccn in all_ccns if ccn <= 10)
        medium = sum(1 for ccn in all_ccns if 10 < ccn <= 20)
        high   = sum(1 for ccn in all_ccns if ccn > 20)

        print(f"\n Complexity Distribution:")
        print(f"   Low    (CCN ≤ 10):       {low}  ({low/len(all_ccns)*100:.1f}%)")
        print(f"   Medium (10 < CCN ≤ 20):  {medium}  ({medium/len(all_ccns)*100:.1f}%)")
        print(f"   High   (CCN > 20):       {high}  ({high/len(all_ccns)*100:.1f}%)")

print(f"\n Results saved to: {OUTPUT_FILE}")
print("\n Analysis complete!")



> Script to examine Lizard rejections with verbose error messages.



In [ ]:
# one-off debug cell
import subprocess

test_file = 'PATH TO .java file'

# check encoding
with open(test_file, 'rb') as f:
    raw = f.read(100)
print("Raw bytes:", raw[:50])

# run lizard with verbose
result = subprocess.run(['lizard', '-V', test_file], capture_output=True, text=True)
print("stdout:", result.stdout[:500])
print("stderr:", result.stderr[:500])
print("returncode:", result.returncode)

# print the actual java code
with open(test_file, 'r', errors='replace') as f:
    print("\nFile contents:\n", f.read())